## Experiment 1: BSM Call - MLP Policy Gradient - no Transaction Cost - Abs Diff

In [ ]:
%cd ..

import numpy as np
import torch
import torch.nn as nn
from hedging.envs import HedgeCallBS
from hedging.reward_utils import compute_discounted_cumsum_rewards
from hedging.plot_utils import plot_portfolio_vs_option_price
from hedging.tanh_normal import TanhNorm

In [3]:
class PolicyNetwork(nn.Module):
    def __init__(
        self, 
        input_dim, 
        hidden_size, 
        action_dim=1, 
        log_std_min=-20, 
        log_std_max=2
    ):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_size)
        self.fc_mu = nn.Linear(hidden_size, action_dim)
        
        
        self.fc_log_std = nn.Linear(hidden_size, action_dim)
        
        self.action_dim = action_dim
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max

    def forward(self, history_features):
        x = history_features[:, -1, :]
        x = torch.tanh(self.fc1(x))
        mu = self.fc_mu(x)  # Directly bound mu
        log_std = self.fc_log_std(x)
        log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)
        return mu, log_std

    def sample_action(self, mu, dist_params, deterministic=False):
        
        log_std = dist_params
        std = torch.exp(log_std)
        distribution = TanhNorm(mu, std)           
        if deterministic:
            action = torch.tanh(mu)
        else:
            action = distribution.sample()
        log_prob = distribution.log_prob(action)
        
        return action, log_prob


In [4]:
class InactionNet(nn.Module):
    def __init__(
        self, 
        input_dim, 
        hidden_size, 
        action_dim=1, 
        log_std_min=-20, 
        log_std_max=2
    ):
        super(InactionNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_size)
        self.fc_mu = nn.Linear(hidden_size, action_dim)
        
        
        self.fc_log_std = nn.Linear(hidden_size, action_dim)
        
        self.action_dim = action_dim
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max

    def forward(self, history_features):
        x = history_features[:, -1, :]
        x = torch.tanh(self.fc1(x))
        mu = self.fc_mu(x)  # Directly bound mu
        log_std = self.fc_log_std(x)
        log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)
        return mu, log_std

    def sample_action(self, mu, dist_params, deterministic=False):
        
        log_std = dist_params
        std = torch.exp(log_std)
        distribution = TanhNorm(mu, std)           
        if deterministic:
            action = torch.tanh(mu)
        else:
            action = distribution.sample()
        log_prob = distribution.log_prob(action)
        
        return action, log_prob


In [4]:
# --- Env. Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
maturity = 1.0
r = 0.05
sigma = np.array([0.15, 0.2, 0.25])
num_paths = 100
num_steps = 250
history_len = 1
transaction_cost = False # No transaction costs are being considered.
transaction_fee_rate = 0.1 

env = HedgeCallBS(
    S0, K, maturity, r, sigma, num_paths, num_steps, history_len=history_len, 
                      transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate
)

# --- Policy Network Parameters ---
input_dim = 11
hidden_size = 64

policy_net = PolicyNetwork(input_dim, hidden_size)
inaction_net = InactionNet(input_dim, hidden_size)

# --- Optimization Parameters ---
learning_rate = 5*1e-4 # Seems to work best for now: 1e-3 was a bit unstable with the dual network architecture

optimizer = torch.optim.Adam(policy_net.parameters(), lr=learning_rate)
optimizer_2 = torch.optim.Adam(inaction_net.parameters(), lr=learning_rate)

# --- Other Parameters ---
num_episodes = 200
num_epochs = 20
discount_factor = 0.999

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net.to(device)
inaction_net.to(device)

for epoch in range(num_epochs):
    for episode in range(num_episodes):
        log_prob_history = []
        inaction_log_prob_history = []  # ADD: Track inaction decisions
        reward_history = []
        state_history = []

        state, _ = env.reset(seed=epoch + 1000)  # [num_envs, obs_dim]
        state_history.append(state)

        done = np.zeros(env.num_envs, dtype=bool)

        while not all(done):
            # Prepare input to policy and inaction net
            policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
            policy_net_input_tensor = torch.tensor(policy_net_input, dtype=torch.float32).to(device)

            # Get action
            action_mu, action_sigma = policy_net(policy_net_input_tensor)
            action, log_prob = policy_net.sample_action(action_mu, action_sigma)
            log_prob_history.append(log_prob)
            action_np = action.detach().cpu().numpy()

            # Decide which envs to step using inaction_net
            inaction_mu, inaction_sigma = inaction_net(policy_net_input_tensor)
            inaction_action, inaction_log_prob = inaction_net.sample_action(inaction_mu, inaction_sigma)
            inaction_log_prob_history.append(inaction_log_prob)
            
            # Convert inaction action to boolean decision (> 0.5 means take action)
            should_step = (inaction_action > 0.0).cpu().numpy()  # boolean mask

            # Step all environments
            next_state_all, reward_all, done_all, _, _ = env.step(action_np)

            # Initialize new buffers
            next_state = np.copy(state[:, 0, :])  # [num_envs, obs_dim]
            reward = np.zeros(env.num_envs)
            new_done = np.copy(done)

            # Apply step results only where allowed
            for i in range(env.num_envs):
                if not done[i]:
                    if should_step[i]:
                        next_state[i] = next_state_all[i]
                        reward[i] = reward_all[i]
                        new_done[i] = done_all[i]
                    else:
                        # Keep state, give same reward as if stepped, but still check if episode should end
                        reward[i] = reward_all[i]
                        new_done[i] = done_all[i]

            # Update trackers
            state = next_state[:, None, :]
            state_history.append(state)
            reward_history.append(reward)
            done = new_done

        # Compute and normalize rewards
        R = compute_discounted_cumsum_rewards(np.array(reward_history), discount_factor)  # shape: [time, num_envs]
        R = R - R.mean(axis=1, keepdims=True)
        R = R / (R.std(axis=1, keepdims=True) + np.finfo(R.dtype).eps)
        R = torch.tensor(R, dtype=torch.float32).to(device)

        # Compute loss and update
        optimizer.zero_grad()
        optimizer_2.zero_grad()
        
        # Policy loss
        policy_loss = (-R * torch.stack(log_prob_history)).mean()
        
        # ADD: Inaction loss  
        inaction_loss = (-R * torch.stack(inaction_log_prob_history)).mean()
        
        policy_loss.backward()
        inaction_loss.backward()
        
        optimizer.step()
        optimizer_2.step()

        if (episode + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}/{num_epochs}, "
                f"Episode {episode + 1}/{num_episodes}, "
                f"Policy Loss: {policy_loss.item():.4f}, "
                f"Inaction Loss: {inaction_loss.item():.4f}, "
                f"Avg. Reward: {np.array(reward_history).mean():.4f}"
            )

Epoch 1/20, Episode 10/200, Policy Loss: -0.0012, Inaction Loss: 0.0285, Avg. Reward: -14.5865
Epoch 1/20, Episode 20/200, Policy Loss: -0.0010, Inaction Loss: 0.0475, Avg. Reward: -14.0871
Epoch 1/20, Episode 30/200, Policy Loss: -0.0077, Inaction Loss: 0.0424, Avg. Reward: -13.9190
Epoch 1/20, Episode 40/200, Policy Loss: -0.0057, Inaction Loss: 0.0474, Avg. Reward: -13.7077
Epoch 1/20, Episode 50/200, Policy Loss: -0.0092, Inaction Loss: 0.0590, Avg. Reward: -13.0856
Epoch 1/20, Episode 60/200, Policy Loss: -0.0104, Inaction Loss: 0.0425, Avg. Reward: -12.3688
Epoch 1/20, Episode 70/200, Policy Loss: -0.0123, Inaction Loss: 0.0440, Avg. Reward: -13.3377
Epoch 1/20, Episode 80/200, Policy Loss: -0.0139, Inaction Loss: 0.0473, Avg. Reward: -12.1829
Epoch 1/20, Episode 90/200, Policy Loss: -0.0096, Inaction Loss: 0.0485, Avg. Reward: -11.2958
Epoch 1/20, Episode 100/200, Policy Loss: -0.0079, Inaction Loss: 0.0603, Avg. Reward: -11.0776
Epoch 1/20, Episode 110/200, Policy Loss: -0.0152

In [6]:
action_sigma.min(), action_sigma.max(), action_sigma.mean(), action.min(), action.max(), action.mean()

(tensor(-4.3822, grad_fn=<MinBackward1>),
 tensor(-2.0542, grad_fn=<MaxBackward1>),
 tensor(-3.6431, grad_fn=<MeanBackward0>),
 tensor(-0.4488),
 tensor(0.9807),
 tensor(0.5716))

In [7]:
env = HedgeCallBS(
    S0, K, maturity, r, sigma, 5, num_steps, history_len=history_len, 
                      transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate
)
env.reset(seed=0)

log_prob_history = []
inaction_log_prob_history = []  # ADD: Also track inaction decisions in test
reward_history = []
state_history = []
action_taken_history = []  # Track when actions were actually taken

state, _ = env.reset(seed=0)
state = state[:, None, :]
state_history.append(state)

done = np.zeros(env.num_envs, dtype=bool)

while not all(done):
    policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
    policy_net_input_tensor = torch.tensor(policy_net_input, dtype=torch.float32).to(device)
    
    # Get action from policy network
    action_mu, action_sigma = policy_net(policy_net_input_tensor)
    action, log_prob = policy_net.sample_action(action_mu, action_sigma, True)
    log_prob_history.append(log_prob)
    action_np = action.detach().cpu().numpy()
    
    # Get inaction decision - FIX: Properly unpack the tuple like in training
    inaction_mu, inaction_sigma = inaction_net(policy_net_input_tensor)
    inaction_action, inaction_log_prob = inaction_net.sample_action(inaction_mu, inaction_sigma, True)
    inaction_log_prob_history.append(inaction_log_prob)
    
    # Convert inaction action to boolean decision (> 0.5 means take action)
    should_step = (inaction_action > 0.0).cpu().numpy()  # boolean mask
    action_taken_history.append(should_step.copy())  # Track for later analysis
    
    # Step all environments
    next_state_all, reward_all, done_all, _, _ = env.step(action_np)
    
    # Initialize new buffers
    next_state = np.copy(state[:, 0, :])
    reward = np.zeros(env.num_envs)
    new_done = np.copy(done)
    
    # Apply step results only where allowed
    for i in range(env.num_envs):
        if not done[i]:
            if should_step[i]:
                next_state[i] = next_state_all[i]
                reward[i] = reward_all[i]
                new_done[i] = done_all[i]
            else:
                # Keep state, give same reward as if stepped, but still check if episode should end
                reward[i] = reward_all[i]
                new_done[i] = done_all[i]
    
    # Update trackers
    state = next_state[:, None, :]
    state_history.append(state)
    reward_history.append(reward)
    done = new_done

print(f"Test completed. Total reward: {np.array(reward_history).sum():.4f}")
print(f"Average reward per step: {np.array(reward_history).mean():.4f}")
print(f"Actions taken: {np.mean([step.sum() for step in action_taken_history]):.2f} out of {env.num_envs} environments per step")

Test completed. Total reward: -17079.2733
Average reward per step: -2.2772
Actions taken: 7.35 out of 30 environments per step


In [8]:
rewards = np.array(reward_history)
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

(-24.192588389192284,
 -1.4295813549836112e-05,
 -2.2772364388163604,
 3.028722497770466)

In [9]:
plot_portfolio_vs_option_price(env)

## Experiment 2: BSM Call - MLP Policy Gradient - Transaction Cost - Abs Diff

In [10]:
# --- Env. Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
maturity = 1.0
r = 0.05
sigma = np.array([0.15, 0.2, 0.25])
num_paths = 100
num_steps = 250
history_len = 1
transaction_cost = True 
transaction_fee_rate = 0.001 # Test transaction cost 0.1%

env = HedgeCallBS(
    S0, K, maturity, r, sigma, num_paths, num_steps, history_len=history_len, 
                      transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate
)

# --- Policy Network Parameters ---
input_dim = 11
hidden_size = 64

policy_net = PolicyNetwork(input_dim, hidden_size)
inaction_net = InactionNet(input_dim, hidden_size)

# --- Optimization Parameters ---

learning_rate = 2.5*1e-4 # Seems to work best for now: 1e-3 was a bit unstable with the dual network architecture

optimizer = torch.optim.Adam(policy_net.parameters(), lr=learning_rate)
optimizer_2 = torch.optim.Adam(inaction_net.parameters(), lr=learning_rate)

# --- Other Parameters ---
num_episodes = 200
num_epochs = 20
discount_factor = 0.999

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net.to(device)
inaction_net.to(device)

for epoch in range(num_epochs):
    for episode in range(num_episodes):
        log_prob_history = []
        inaction_log_prob_history = [] 
        reward_history = []
        state_history = []

        state, _ = env.reset(seed=epoch + 1000)  # [num_envs, obs_dim]
        state_history.append(state)

        done = np.zeros(env.num_envs, dtype=bool)

        while not all(done):
            # Prepare input to policy and inaction net
            policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
            policy_net_input_tensor = torch.tensor(policy_net_input, dtype=torch.float32).to(device)

            # Get action
            action_mu, action_sigma = policy_net(policy_net_input_tensor)
            action, log_prob = policy_net.sample_action(action_mu, action_sigma)
            log_prob_history.append(log_prob)
            action_np = action.detach().cpu().numpy()

            # Decide which envs to step using inaction_net
            inaction_mu, inaction_sigma = inaction_net(policy_net_input_tensor)
            inaction_action, inaction_log_prob = inaction_net.sample_action(inaction_mu, inaction_sigma)
            inaction_log_prob_history.append(inaction_log_prob)
            
            # Convert inaction action to boolean decision (> 0.5 means take action)
            should_step = (inaction_action > 0.0).cpu().numpy()  # boolean mask

            # Step all environments
            next_state_all, reward_all, done_all, _, _ = env.step(action_np)

            # Initialize new buffers
            next_state = np.copy(state[:, 0, :])  # [num_envs, obs_dim]
            reward = np.zeros(env.num_envs)
            new_done = np.copy(done)

            # Apply step results only where allowed
            for i in range(env.num_envs):
                if not done[i]:
                    if should_step[i]:
                        next_state[i] = next_state_all[i]
                        reward[i] = reward_all[i]
                        new_done[i] = done_all[i]
                    else:
                        # Keep state, give same reward as if stepped, but still check if episode should end
                        reward[i] = reward_all[i]
                        new_done[i] = done_all[i]

            # Update trackers
            state = next_state[:, None, :]
            state_history.append(state)
            reward_history.append(reward)
            done = new_done
            

        # Compute and normalize rewards
        R = compute_discounted_cumsum_rewards(np.array(reward_history), discount_factor)  # shape: [time, num_envs]
        R = R - R.mean(axis=1, keepdims=True)
        R = R / (R.std(axis=1, keepdims=True) + np.finfo(R.dtype).eps)
        R = torch.tensor(R, dtype=torch.float32).to(device)

        # Compute loss and update
        optimizer.zero_grad()
        optimizer_2.zero_grad()
        
        # Policy loss
        policy_loss = (-R * torch.stack(log_prob_history)).mean()
        
        # ADD: Inaction loss  
        inaction_loss = (-R * torch.stack(inaction_log_prob_history)).mean()
        
        policy_loss.backward()
        inaction_loss.backward()
        
        optimizer.step()
        optimizer_2.step()

        if (episode + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}/{num_epochs}, "
                f"Episode {episode + 1}/{num_episodes}, "
                f"Policy Loss: {policy_loss.item():.4f}, "
                f"Inaction Loss: {inaction_loss.item():.4f}, "
                f"Avg. Reward: {np.array(reward_history).mean():.4f}"
            )

Epoch 1/20, Episode 10/200, Policy Loss: 0.0048, Inaction Loss: 0.0146, Avg. Reward: -14.7437
Epoch 1/20, Episode 20/200, Policy Loss: 0.0071, Inaction Loss: 0.0152, Avg. Reward: -14.3760
Epoch 1/20, Episode 30/200, Policy Loss: 0.0023, Inaction Loss: 0.0160, Avg. Reward: -14.1377
Epoch 1/20, Episode 40/200, Policy Loss: 0.0024, Inaction Loss: 0.0124, Avg. Reward: -13.6559
Epoch 1/20, Episode 50/200, Policy Loss: 0.0008, Inaction Loss: 0.0146, Avg. Reward: -13.7603
Epoch 1/20, Episode 60/200, Policy Loss: -0.0017, Inaction Loss: 0.0159, Avg. Reward: -13.4107
Epoch 1/20, Episode 70/200, Policy Loss: -0.0013, Inaction Loss: 0.0175, Avg. Reward: -12.5822
Epoch 1/20, Episode 80/200, Policy Loss: -0.0057, Inaction Loss: 0.0184, Avg. Reward: -12.2777
Epoch 1/20, Episode 90/200, Policy Loss: -0.0077, Inaction Loss: 0.0144, Avg. Reward: -12.0753
Epoch 1/20, Episode 100/200, Policy Loss: -0.0115, Inaction Loss: 0.0142, Avg. Reward: -11.6571
Epoch 1/20, Episode 110/200, Policy Loss: -0.0146, Ina

In [ ]:
env = HedgeCallBS(
    S0, K, maturity, r, sigma, 5, num_steps, history_len=history_len, 
                      transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate
)
env.reset(seed=1000)

log_prob_history = []
inaction_log_prob_history = []  # ADD: Also track inaction decisions in test
reward_history = []
state_history = []
action_taken_history = []  # Track when actions were actually taken

state, _ = env.reset(seed=1)
state = state[:, None, :]
state_history.append(state)

done = np.zeros(env.num_envs, dtype=bool)

while not all(done):
    policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
    policy_net_input_tensor = torch.tensor(policy_net_input, dtype=torch.float32).to(device)
    
    # Get action from policy network
    action_mu, action_sigma = policy_net(policy_net_input_tensor)
    action, log_prob = policy_net.sample_action(action_mu, action_sigma, True)
    log_prob_history.append(log_prob)
    action_np = action.detach().cpu().numpy()
    
    # Get inaction decision - FIX: Properly unpack the tuple like in training
    inaction_mu, inaction_sigma = inaction_net(policy_net_input_tensor)
    inaction_action, inaction_log_prob = inaction_net.sample_action(inaction_mu, inaction_sigma, True)
    inaction_log_prob_history.append(inaction_log_prob)
    
    # Convert inaction action to boolean decision (> 0.5 means take action)
    should_step = (inaction_action > 0.0).cpu().numpy()  # boolean mask
    action_taken_history.append(should_step.copy())  # Track for later analysis
    
    # Step all environments
    next_state_all, reward_all, done_all, _, _ = env.step(action_np)
    
    # Initialize new buffers
    next_state = np.copy(state[:, 0, :])
    reward = np.zeros(env.num_envs)
    new_done = np.copy(done)
    
    # Apply step results only where allowed
    for i in range(env.num_envs):
        if not done[i]:
            if should_step[i]:
                next_state[i] = next_state_all[i]
                reward[i] = reward_all[i]
                new_done[i] = done_all[i]
            else:
                # Keep state, give same reward as if stepped, but still check if episode should end
                reward[i] = reward_all[i]
                new_done[i] = done_all[i]
    
    # Update trackers
    state = next_state[:, None, :]
    state_history.append(state)
    reward_history.append(reward)
    done = new_done

print(f"Test completed. Total reward: {np.array(reward_history).sum():.4f}")
print(f"Average reward per step: {np.array(reward_history).mean():.4f}")
print(f"Actions taken: {np.mean([step.sum() for step in action_taken_history]):.2f} out of {env.num_envs} environments per step")

Test completed. Total reward: -29986.0866
Average reward per step: -3.9981
Actions taken: 0.00 out of 30 environments per step


In [19]:
action_sigma.min(), action_sigma.max(), action_sigma.mean(), action.min(), action.max(), action.mean()

(tensor(-3.9736, grad_fn=<MinBackward1>),
 tensor(-3.9605, grad_fn=<MaxBackward1>),
 tensor(-3.9671, grad_fn=<MeanBackward0>),
 tensor(0.4675, grad_fn=<MinBackward1>),
 tensor(0.5917, grad_fn=<MaxBackward1>),
 tensor(0.5296, grad_fn=<MeanBackward0>))

In [20]:
rewards = np.array(reward_history)
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

(-30.149978051982217,
 -0.0004316420678742716,
 -3.9981448852582244,
 4.748313892864322)

In [21]:
plot_portfolio_vs_option_price(env)

In [7]:
action_1 = []
reward_1 = []
tot_reward1 = []
action_2 = []
reward_2 = []
tot_reward2 = []

for i in range(10):
    print(f"Starting Simulation: {i+1}/10")
    # --- Env. Parameters ---
    S0 = np.array([50.0, 100.0, 200.0])
    K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
    maturity = 1.0
    r = 0.05
    sigma = np.array([0.15, 0.2, 0.25])
    num_paths = 100
    num_steps = 250
    history_len = 1
    transaction_cost = False # No transaction costs are being considered.
    transaction_fee_rate = 0.1 

    env = HedgeCallBS(
        S0, K, maturity, r, sigma, num_paths, num_steps, history_len=history_len, 
                        transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate
    )

    # --- Policy Network Parameters ---
    input_dim = 11
    hidden_size = 64

    policy_net = PolicyNetwork(input_dim, hidden_size)
    inaction_net = InactionNet(input_dim, hidden_size)

    # --- Optimization Parameters ---
    learning_rate = 5*1e-4 # Seems to work best for now: 1e-3 was a bit unstable with the dual network architecture

    optimizer = torch.optim.Adam(policy_net.parameters(), lr=learning_rate)
    optimizer_2 = torch.optim.Adam(inaction_net.parameters(), lr=learning_rate)

    # --- Other Parameters ---
    num_episodes = 200
    num_epochs = 20
    discount_factor = 0.999

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    policy_net.to(device)
    inaction_net.to(device)

    for epoch in range(num_epochs):
        for episode in range(num_episodes):
            log_prob_history = []
            inaction_log_prob_history = []  # ADD: Track inaction decisions
            reward_history = []
            state_history = []

            state, _ = env.reset(seed=epoch + 1000)  # [num_envs, obs_dim]
            state_history.append(state)

            done = np.zeros(env.num_envs, dtype=bool)

            while not all(done):
                # Prepare input to policy and inaction net
                policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
                policy_net_input_tensor = torch.tensor(policy_net_input, dtype=torch.float32).to(device)

                # Get action
                action_mu, action_sigma = policy_net(policy_net_input_tensor)
                action, log_prob = policy_net.sample_action(action_mu, action_sigma)
                log_prob_history.append(log_prob)
                action_np = action.detach().cpu().numpy()

                # Decide which envs to step using inaction_net
                inaction_mu, inaction_sigma = inaction_net(policy_net_input_tensor)
                inaction_action, inaction_log_prob = inaction_net.sample_action(inaction_mu, inaction_sigma)
                inaction_log_prob_history.append(inaction_log_prob)
                
                # Convert inaction action to boolean decision (> 0.5 means take action)
                should_step = (inaction_action > 0.0).cpu().numpy()  # boolean mask

                # Step all environments
                next_state_all, reward_all, done_all, _, _ = env.step(action_np)

                # Initialize new buffers
                next_state = np.copy(state[:, 0, :])  # [num_envs, obs_dim]
                reward = np.zeros(env.num_envs)
                new_done = np.copy(done)

                # Apply step results only where allowed
                for i in range(env.num_envs):
                    if not done[i]:
                        if should_step[i]:
                            next_state[i] = next_state_all[i]
                            reward[i] = reward_all[i]
                            new_done[i] = done_all[i]
                        else:
                            # Keep state, give same reward as if stepped, but still check if episode should end
                            reward[i] = reward_all[i]
                            new_done[i] = done_all[i]

                # Update trackers
                state = next_state[:, None, :]
                state_history.append(state)
                reward_history.append(reward)
                done = new_done

            # Compute and normalize rewards
            R = compute_discounted_cumsum_rewards(np.array(reward_history), discount_factor)  # shape: [time, num_envs]
            R = R - R.mean(axis=1, keepdims=True)
            R = R / (R.std(axis=1, keepdims=True) + np.finfo(R.dtype).eps)
            R = torch.tensor(R, dtype=torch.float32).to(device)

            # Compute loss and update
            optimizer.zero_grad()
            optimizer_2.zero_grad()
            
            # Policy loss
            policy_loss = (-R * torch.stack(log_prob_history)).mean()
            
            # ADD: Inaction loss  
            inaction_loss = (-R * torch.stack(inaction_log_prob_history)).mean()
            
            policy_loss.backward()
            inaction_loss.backward()
            
            optimizer.step()
            optimizer_2.step()

            if (episode + 1) % 10 == 0:
                print(
                    f"Epoch {epoch+1}/{num_epochs}, "
                    f"Episode {episode + 1}/{num_episodes}, "
                    f"Policy Loss: {policy_loss.item():.4f}, "
                    f"Inaction Loss: {inaction_loss.item():.4f}, "
                    f"Avg. Reward: {np.array(reward_history).mean():.4f}"
                )
    env = HedgeCallBS(
    S0, K, maturity, r, sigma, 5, num_steps, history_len=history_len, 
                      transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate
    )
    env.reset(seed=0)

    log_prob_history = []
    inaction_log_prob_history = []  # ADD: Also track inaction decisions in test
    reward_history = []
    state_history = []
    action_taken_history = []  # Track when actions were actually taken

    state, _ = env.reset(seed=0)
    state = state[:, None, :]
    state_history.append(state)

    done = np.zeros(env.num_envs, dtype=bool)

    while not all(done):
        policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
        policy_net_input_tensor = torch.tensor(policy_net_input, dtype=torch.float32).to(device)
        
        # Get action from policy network
        action_mu, action_sigma = policy_net(policy_net_input_tensor)
        action, log_prob = policy_net.sample_action(action_mu, action_sigma, True)
        log_prob_history.append(log_prob)
        action_np = action.detach().cpu().numpy()
        
        # Get inaction decision - FIX: Properly unpack the tuple like in training
        inaction_mu, inaction_sigma = inaction_net(policy_net_input_tensor)
        inaction_action, inaction_log_prob = inaction_net.sample_action(inaction_mu, inaction_sigma, True)
        inaction_log_prob_history.append(inaction_log_prob)
        
    
        should_step = (inaction_action > 0.0).cpu().numpy()  # boolean mask (0.0 > Take action)
        action_taken_history.append(should_step.copy())  # Track for later analysis
        
        # Step all environments
        next_state_all, reward_all, done_all, _, _ = env.step(action_np)
        
        # Initialize new buffers
        next_state = np.copy(state[:, 0, :])
        reward = np.zeros(env.num_envs)
        new_done = np.copy(done)
        
        # Apply step results only where allowed
        for i in range(env.num_envs):
            if not done[i]:
                if should_step[i]:
                    next_state[i] = next_state_all[i]
                    reward[i] = reward_all[i]
                    new_done[i] = done_all[i]
                else:
                    # Keep state, give same reward as if stepped, but still check if episode should end
                    reward[i] = reward_all[i]
                    new_done[i] = done_all[i]
        
        # Update trackers
        state = next_state[:, None, :]
        state_history.append(state)
        reward_history.append(reward)
        done = new_done

    print(f"Test completed. Total reward: {np.array(reward_history).sum():.4f}")
    print(f"Average reward per step: {np.array(reward_history).mean():.4f}")
    print(f"Actions taken: {np.mean([step.sum() for step in action_taken_history]):.2f} out of {env.num_envs} environments per step")
    tot_reward1.append(np.array(reward_history).sum())
    reward_1.append(np.array(reward_history).mean())
    action_1.append(np.mean([step.sum() for step in action_taken_history]))

    # --- Env. Parameters ---
    S0 = np.array([50.0, 100.0, 200.0])
    K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
    maturity = 1.0
    r = 0.05
    sigma = np.array([0.15, 0.2, 0.25])
    num_paths = 100
    num_steps = 250
    history_len = 1
    transaction_cost = True # No transaction costs are being considered.
    transaction_fee_rate = 0.001 

    env = HedgeCallBS(
        S0, K, maturity, r, sigma, num_paths, num_steps, history_len=history_len, 
                        transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate
    )

    # --- Policy Network Parameters ---
    input_dim = 11
    hidden_size = 64

    policy_net = PolicyNetwork(input_dim, hidden_size)
    inaction_net = InactionNet(input_dim, hidden_size)

    # --- Optimization Parameters ---
    learning_rate = 5*1e-4 # Seems to work best for now: 1e-3 was a bit unstable with the dual network architecture

    optimizer = torch.optim.Adam(policy_net.parameters(), lr=learning_rate)
    optimizer_2 = torch.optim.Adam(inaction_net.parameters(), lr=learning_rate)

    # --- Other Parameters ---
    num_episodes = 200
    num_epochs = 20
    discount_factor = 0.999

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    policy_net.to(device)
    inaction_net.to(device)

    for epoch in range(num_epochs):
        for episode in range(num_episodes):
            log_prob_history = []
            inaction_log_prob_history = []  # ADD: Track inaction decisions
            reward_history = []
            state_history = []

            state, _ = env.reset(seed=epoch + 1000)  # [num_envs, obs_dim]
            state_history.append(state)

            done = np.zeros(env.num_envs, dtype=bool)

            while not all(done):
                # Prepare input to policy and inaction net
                policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
                policy_net_input_tensor = torch.tensor(policy_net_input, dtype=torch.float32).to(device)

                # Get action
                action_mu, action_sigma = policy_net(policy_net_input_tensor)
                action, log_prob = policy_net.sample_action(action_mu, action_sigma)
                log_prob_history.append(log_prob)
                action_np = action.detach().cpu().numpy()

                # Decide which envs to step using inaction_net
                inaction_mu, inaction_sigma = inaction_net(policy_net_input_tensor)
                inaction_action, inaction_log_prob = inaction_net.sample_action(inaction_mu, inaction_sigma)
                inaction_log_prob_history.append(inaction_log_prob)
                
                # Convert inaction action to boolean decision (> 0.5 means take action)
                should_step = (inaction_action > 0.0).cpu().numpy()  # boolean mask

                # Step all environments
                next_state_all, reward_all, done_all, _, _ = env.step(action_np)

                # Initialize new buffers
                next_state = np.copy(state[:, 0, :])  # [num_envs, obs_dim]
                reward = np.zeros(env.num_envs)
                new_done = np.copy(done)

                # Apply step results only where allowed
                for i in range(env.num_envs):
                    if not done[i]:
                        if should_step[i]:
                            next_state[i] = next_state_all[i]
                            reward[i] = reward_all[i]
                            new_done[i] = done_all[i]
                        else:
                            # Keep state, give same reward as if stepped, but still check if episode should end
                            reward[i] = reward_all[i]
                            new_done[i] = done_all[i]

                # Update trackers
                state = next_state[:, None, :]
                state_history.append(state)
                reward_history.append(reward)
                done = new_done

            # Compute and normalize rewards
            R = compute_discounted_cumsum_rewards(np.array(reward_history), discount_factor)  # shape: [time, num_envs]
            R = R - R.mean(axis=1, keepdims=True)
            R = R / (R.std(axis=1, keepdims=True) + np.finfo(R.dtype).eps)
            R = torch.tensor(R, dtype=torch.float32).to(device)

            # Compute loss and update
            optimizer.zero_grad()
            optimizer_2.zero_grad()
            
            # Policy loss
            policy_loss = (-R * torch.stack(log_prob_history)).mean()
            
            # ADD: Inaction loss  
            inaction_loss = (-R * torch.stack(inaction_log_prob_history)).mean()
            
            policy_loss.backward()
            inaction_loss.backward()
            
            optimizer.step()
            optimizer_2.step()

            if (episode + 1) % 10 == 0:
                print(
                    f"Epoch {epoch+1}/{num_epochs}, "
                    f"Episode {episode + 1}/{num_episodes}, "
                    f"Policy Loss: {policy_loss.item():.4f}, "
                    f"Inaction Loss: {inaction_loss.item():.4f}, "
                    f"Avg. Reward: {np.array(reward_history).mean():.4f}"
                )
    env = HedgeCallBS(
    S0, K, maturity, r, sigma, 5, num_steps, history_len=history_len, 
                      transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate
    )
    env.reset(seed=0)

    log_prob_history = []
    inaction_log_prob_history = []  # ADD: Also track inaction decisions in test
    reward_history = []
    state_history = []
    action_taken_history = []  # Track when actions were actually taken

    state, _ = env.reset(seed=0)
    state = state[:, None, :]
    state_history.append(state)

    done = np.zeros(env.num_envs, dtype=bool)

    while not all(done):
        policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
        policy_net_input_tensor = torch.tensor(policy_net_input, dtype=torch.float32).to(device)
        
        # Get action from policy network
        action_mu, action_sigma = policy_net(policy_net_input_tensor)
        action, log_prob = policy_net.sample_action(action_mu, action_sigma, True)
        log_prob_history.append(log_prob)
        action_np = action.detach().cpu().numpy()
        
        # Get inaction decision - FIX: Properly unpack the tuple like in training
        inaction_mu, inaction_sigma = inaction_net(policy_net_input_tensor)
        inaction_action, inaction_log_prob = inaction_net.sample_action(inaction_mu, inaction_sigma, True)
        inaction_log_prob_history.append(inaction_log_prob)
        
        # Convert inaction action to boolean decision (> 0.5 means take action)
        should_step = (inaction_action > 0.0).cpu().numpy()  # boolean mask
        action_taken_history.append(should_step.copy())  # Track for later analysis
        
        # Step all environments
        next_state_all, reward_all, done_all, _, _ = env.step(action_np)
        
        # Initialize new buffers
        next_state = np.copy(state[:, 0, :])
        reward = np.zeros(env.num_envs)
        new_done = np.copy(done)
        
        # Apply step results only where allowed
        for i in range(env.num_envs):
            if not done[i]:
                if should_step[i]:
                    next_state[i] = next_state_all[i]
                    reward[i] = reward_all[i]
                    new_done[i] = done_all[i]
                else:
                    # Keep state, give same reward as if stepped, but still check if episode should end
                    reward[i] = reward_all[i]
                    new_done[i] = done_all[i]
        
        # Update trackers
        state = next_state[:, None, :]
        state_history.append(state)
        reward_history.append(reward)
        done = new_done

    print(f"Test completed. Total reward: {np.array(reward_history).sum():.4f}")
    print(f"Average reward per step: {np.array(reward_history).mean():.4f}")
    print(f"Actions taken: {np.mean([step.sum() for step in action_taken_history]):.2f} out of {env.num_envs} environments per step")
    tot_reward2.append(np.array(reward_history).sum())
    reward_2.append(np.array(reward_history).mean())
    action_2.append(np.mean([step.sum() for step in action_taken_history]))

    print(f"End Simulation: {i+1}/10")

    


Starting Simulation: 1/10
Epoch 1/20, Episode 10/200, Policy Loss: 0.0059, Inaction Loss: 0.0169, Avg. Reward: -11.6420
Epoch 1/20, Episode 20/200, Policy Loss: 0.0035, Inaction Loss: 0.0267, Avg. Reward: -11.5761
Epoch 1/20, Episode 30/200, Policy Loss: -0.0013, Inaction Loss: 0.0312, Avg. Reward: -10.7173
Epoch 1/20, Episode 40/200, Policy Loss: -0.0048, Inaction Loss: 0.0361, Avg. Reward: -10.7048
Epoch 1/20, Episode 50/200, Policy Loss: 0.0022, Inaction Loss: 0.0332, Avg. Reward: -9.0643
Epoch 1/20, Episode 60/200, Policy Loss: 0.0098, Inaction Loss: 0.0323, Avg. Reward: -8.5897
Epoch 1/20, Episode 70/200, Policy Loss: 0.0118, Inaction Loss: 0.0344, Avg. Reward: -7.8890
Epoch 1/20, Episode 80/200, Policy Loss: 0.0245, Inaction Loss: 0.0382, Avg. Reward: -7.4298
Epoch 1/20, Episode 90/200, Policy Loss: 0.0197, Inaction Loss: 0.0370, Avg. Reward: -7.0126
Epoch 1/20, Episode 100/200, Policy Loss: 0.0320, Inaction Loss: 0.0516, Avg. Reward: -6.5525
Epoch 1/20, Episode 110/200, Policy L